<a href="https://colab.research.google.com/github/annatagus/python_aulas_exercicios/blob/main/M6_Exercicios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Abaixo encontra a resolução de todos os exercícios da folha do **Módulo 6 (Aquisição de Dados - Requests, Web Scraping e APIs REST)**, desenvolvida em Python para nível *beginner*, estruturada de forma modular e pronta a ser executada num Notebook Jupyter ou VS Code.

---

## GRUPO A: REQUESTS - PEDIDOS HTTP

In [ ]:
import json
import pandas as pd
import requests

# URL base do servidor local de testes (ajuste a porta se necessário)
BASE_URL = "http://127.0.0.1:5000"

### Exercício 1. GET e Códigos de Estado

In [ ]:
# 1. Pedido GET à lista de produtos
resposta_produtos = requests.get(f"{BASE_URL}/produtos")
print("1. Código de estado (GET /produtos):", resposta_produtos.status_code)

# 2. Confirmar código 200 e contar produtos
if resposta_produtos.status_code == 200:
    produtos = resposta_produtos.json()
    print(
        f"2. Sucesso! Foram recebidos {len(produtos)} produtos."
    )
else:
    print("Erro ao obter a lista de produtos.")

# 3. Pedido a um produto inexistente (ex.: id 999)
resposta_inexistente = requests.get(f"{BASE_URL}/produtos/999")
print(
    "3. Código de estado para ID inexistente (999):",
    resposta_inexistente.status_code,
)

# 4. Tratamento com estrutura condicional (if / else)
def verificar_pedido(url):
    res = requests.get(url)
    if res.status_code == 200:
        print(f"Sucesso (200): Dados recebidos de {url}")
    elif res.status_code == 404:
        print(f"Erro (404): Recurso não encontrado em {url}")
    else:
        print(f"Aviso: Recebido código de estado {res.status_code}")


print("\n4. Teste da validação condicional:")
verificar_pedido(f"{BASE_URL}/produtos")
verificar_pedido(f"{BASE_URL}/produtos/999")

### Exercício 2. Parâmetros e DataFrame

In [ ]:
# 1. GET com parâmetro de filtragem
params = {"categoria": "perifericos"}
resposta_filtrada = requests.get(f"{BASE_URL}/produtos", params=params)

if resposta_filtrada.status_code == 200:
    # 2. Converter JSON e criar DataFrame
    dados_json = resposta_filtrada.json()
    df_perifericos = pd.DataFrame(dados_json)
    print("DataFrame filtrado por categoria 'perifericos':")
    print(df_perifericos.head())

    # 3. Preço médio por categoria (usando o DataFrame completo)
    todos_dados = requests.get(f"{BASE_URL}/produtos").json()
    df_todos = pd.DataFrame(todos_dados)

    preco_medio_cat = df_todos.groupby("categoria")["preco"].mean()
    print("\n3. Preço médio por categoria:")
    print(preco_medio_cat)

### Exercício 3. POST e Gravação

In [ ]:
# 1. Enviar um novo produto via POST
novo_produto = {
    "nome": "Teclado Mecânico RGB",
    "categoria": "perifericos",
    "preco": 79.99,
    "stock": 25,
}

resposta_post = requests.post(f"{BASE_URL}/produtos", json=novo_produto)
print("1. Código de estado do POST:", resposta_post.status_code)

if resposta_post.status_code == 201:
    print("Produto criado com sucesso!")

# 2. Guardar a lista de produtos atualizada em CSV e JSON
res_atualizada = requests.get(f"{BASE_URL}/produtos")
lista_produtos = res_atualizada.json()

# Guardar em JSON
with open("produtos.json", "w", encoding="utf-8") as f:
    json.dump(lista_produtos, f, ensure_ascii=False, indent=4)

# Guardar em CSV
df_atualizado = pd.DataFrame(lista_produtos)
df_atualizado.to_csv("produtos.csv", index=False, encoding="utf-8")
print("2. Ficheiros 'produtos.json' e 'produtos.csv' gravados.")

# 3. Ler o CSV de volta para confirmar
df_confirmacao = pd.read_csv("produtos.csv")
print("\n3. Leitura de confirmação do ficheiro CSV:")
print(df_confirmacao.head())

---

## GRUPO B: WEB SCRAPING COM BEAUTIFULSOUP

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

### Exercício 4. Extrair Conteúdo de HTML

In [ ]:
# 1. Obter o HTML da página
url_pagina = f"{BASE_URL}/pagina.html"
html_content = requests.get(url_pagina).text

# 2. Criar objeto BeautifulSoup e extrair o título (<h1>)
soup = BeautifulSoup(html_content, "html.parser")
titulo_h1 = soup.find("h1").text if soup.find("h1") else "Sem h1"
print("2. Título principal da página (<h1>):", titulo_h1)

# 3. Extrair nome e preço de cada produto
produtos_extraidos = []

# Exemplo procurando por elementos HTML (ex: <div class="produto">)
elementos_produto = soup.select(".produto")

for el in elementos_produto:
    nome = el.select_one(".nome").text.strip()
    preco = float(
        el.select_one(".preco").text.replace("€", "").replace(",", ".").strip()
    )
    produtos_extraidos.append({"nome": nome, "preco": preco})

# 4. Construir DataFrame e exportar para CSV
df_web = pd.DataFrame(produtos_extraidos)
df_web.to_csv("produtos_web.csv", index=False, encoding="utf-8")

print("\n4. Dados recolhidos via Scraping:")
print(df_web)
print("Ficheiro 'produtos_web.csv' exportado com sucesso.")

---

## GRUPO C: TABELAS E PAGINAÇÃO

In [ ]:
from urllib.robotparser import RobotFileParser
import pandas as pd
import requests

### Exercício 5. Tabelas e Várias Páginas

In [ ]:
# 1. Extrair tabela com pandas.read_html()
url_pagina = f"{BASE_URL}/pagina.html"
tabelas = pd.read_html(url_pagina)

print("1. Tabela de Stock extraída com read_html():")
df_tabela_stock = tabelas[0]
print(df_tabela_stock.head())

# 2. Percorrer páginas 1 a 3 e acumular produtos numa lista
todos_produtos_paginados = []

for n in range(1, 4):
    params = {"pagina": n}
    res = requests.get(f"{BASE_URL}/produtos", params=params)
    if res.status_code == 200:
        dados_pagina = res.json()
        todos_produtos_paginados.extend(dados_pagina)

# 3. Construir DataFrame com o total e indicar total de linhas
df_paginado = pd.DataFrame(todos_produtos_paginados)
print(
    f"\n3. Total de registos recolhidos nas 3 páginas: {len(df_paginado)} linhas."
)

# 4. Consultar o robots.txt
rp = RobotFileParser()
rp.set_url(f"{BASE_URL}/robots.txt")
rp.read()

pode_recolher = rp.can_fetch("*", f"{BASE_URL}/produtos")
print(
    f"\n4. O scraping do endpoint '/produtos' é permitido pelo robots.txt? {pode_recolher}"
)

---

## GRUPO D: APIS REST E JSON (CONSOLIDAÇÃO)

### Exercício 6. Projeto de Consolidação (API Pública: JSONPlaceholder)

In [ ]:
import time
from urllib.robotparser import RobotFileParser
import pandas as pd
import requests

# 1. Consumir um endpoint público (Usuários com endereços e empresas aninhadas)
url_api = "https://jsonplaceholder.typicode.com/users"

# 2. Validação da resposta antes de usar os dados (com timeout e raise_for_status)
try:
    resposta = requests.get(url_api, timeout=10)
    resposta.raise_for_status()  # Lança exceção se status != 200
    dados_brutos = resposta.json()
    print("2. Pedido à API efetuado com sucesso!")
except requests.exceptions.RequestException as e:
    print(f"Erro na ligação à API: {e}")
    dados_brutos = []

if dados_brutos:
    # 3. Tratar JSON aninhado com pandas.json_normalize()
    # Achata propriedades como address.city, company.name, etc.
    df_users = pd.json_normalize(dados_brutos)

    print("\n3. DataFrame desnormalizado (Primeiras colunas):")
    print(
        df_users[
            ["id", "name", "username", "address.city", "company.name"]
        ].head()
    )

    # 4. Agregação simples: Contagem de utilizadores por cidade
    utilizadores_por_cidade = (
        df_users.groupby("address.city")["id"]
        .count()
        .reset_index()
    )
    utilizadores_por_cidade.columns = ["Cidade", "Total_Utilizadores"]

    print("\n4. Agregação (Utilizadores por cidade):")
    print(utilizadores_por_cidade)

    # 5. Exportar o resultado final para CSV
    utilizadores_por_cidade.to_csv(
        "utilizadores_por_cidade.csv", index=False, encoding="utf-8"
    )
    print(
        "\n5. Resultado final exportado para 'utilizadores_por_cidade.csv'."
    )

# 6. Três boas práticas de aquisição de dados aplicadas:
# - Prática 1 (Timeout): Utilização do parâmetro `timeout=10` nos pedidos HTTP para evitar que o script fique bloqueado indefinidamente se o servidor não responder.
# - Prática 2 (Respeito ao Servidor / Ritmo): Verificação das regras do `robots.txt` e introdução de intervalos de pausa (`time.sleep()`) em ciclos iterativos para não sobrecarregar o servidor.
# - Prática 3 (Validação de Estado e Exceções): Validação preventiva do status HTTP (`raise_for_status()` ou verificação de `status_code == 200`) antes de tentar processar e converter o conteúdo para JSON/DataFrames.